**Practical 5. Evaluate the Challenges in Building a Context-Aware Chatbot. Suggest Suitable NLP Models for Intent Detection.**

**What is a Context-Aware Chatbot?**

A context-aware chatbot is a conversational agent that understands and remembers the history of a conversation to give relevant, coherent replies — not just reacting to one message at a time.

**What is Intent Detection?**

Intent Detection is the task of identifying the purpose or goal behind a user's message.

Examples:
- "Book me a flight to Delhi" -> Intent: **book_flight**
- "What is the weather today?" -> Intent: **get_weather**
- "Cancel my order" -> Intent: **cancel_order**
- "Hello, how are you?" -> Intent: **greeting**

**Key Components of a Chatbot Pipeline:**

User Input -> **NLU (Intent + Entity Detection)** -> **Dialogue Manager (Context)** -> **NLG (Response Generation)** -> Bot Reply

1. **Import required libraries**

In [16]:
import nltk
import warnings
import random
import numpy as np
import string
warnings.filterwarnings('ignore')

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict

print("All libraries imported successfully!")

All libraries imported successfully!


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2. **Define the Intent Dataset**

Each entry has a user message and its corresponding intent label.
This is our training data for the intent detection model.

In [17]:
intent_data = {
    "greeting": [
        "hello there",
        "hi how are you",
        "hey good morning",
        "good evening",
        "hi there what's up",
        "hello bot",
        "hey how is it going",
        "good afternoon"
    ],
    "goodbye": [
        "bye see you later",
        "goodbye take care",
        "see you soon",
        "have a nice day",
        "talk to you later",
        "I am leaving now",
        "catch you later",
        "good night"
    ],
    "book_flight": [
        "book a flight to Mumbai",
        "I want to fly to Delhi",
        "reserve a seat on the next flight",
        "I need a ticket to Bangalore",
        "book me a flight for tomorrow",
        "can you book a flight to Chennai",
        "I want to travel to Pune by air",
        "get me a ticket to Kolkata"
    ],
    "get_weather": [
        "what is the weather today",
        "will it rain tomorrow",
        "tell me the weather forecast",
        "how is the weather in Delhi",
        "is it going to snow today",
        "what is the temperature outside",
        "give me today's weather update",
        "how hot is it in Mumbai"
    ],
    "cancel_order": [
        "cancel my order please",
        "I want to cancel my booking",
        "please cancel my flight ticket",
        "how do I cancel my reservation",
        "I need to cancel my purchase",
        "cancel the order I just placed",
        "I changed my mind cancel it",
        "please remove my order"
    ],
    "check_order_status": [
        "where is my order",
        "what is the status of my booking",
        "track my order please",
        "when will my order arrive",
        "has my package been shipped",
        "what is the delivery status",
        "I want to track my shipment",
        "tell me the order delivery date"
    ]
}

print("Intent Dataset:")
print()
for intent, examples in intent_data.items():
    print(f"Intent: '{intent}'  ->  {len(examples)} examples")

Intent Dataset:

Intent: 'greeting'  ->  8 examples
Intent: 'goodbye'  ->  8 examples
Intent: 'book_flight'  ->  8 examples
Intent: 'get_weather'  ->  8 examples
Intent: 'cancel_order'  ->  8 examples
Intent: 'check_order_status'  ->  8 examples


3. **Prepare training data -> Flatten into (text, label) pairs**

In [18]:
texts = []
labels = []

for intent, examples in intent_data.items():
    for example in examples:
        texts.append(example)
        labels.append(intent)

print("Total training samples  :  ", len(texts))
print("Total intent classes    :  ", len(set(labels)))
print()
print("Sample (text, label) pairs:")
print()
for t, l in list(zip(texts, labels))[:6]:
    print(f"'{t}'  :  {l}")

Total training samples  :   48
Total intent classes    :   6

Sample (text, label) pairs:

'hello there'  :  greeting
'hi how are you'  :  greeting
'hey good morning'  :  greeting
'good evening'  :  greeting
'hi there what's up'  :  greeting
'hello bot'  :  greeting


4. **Preprocess text -> Lowercase, remove stopwords, lemmatize**

Method -> preprocess(text) returns a clean string

In [19]:
lemmatizer = WordNetLemmatizer()
sw = set(stopwords.words('english'))

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t not in string.punctuation]
    tokens = [t for t in tokens if t not in sw]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

processed_texts = [preprocess(t) for t in texts]

print("Preprocessed Texts (first 6):")
print()
for original, processed in list(zip(texts, processed_texts))[:6]:
    print(f"Original   :  '{original}'")
    print(f"Processed  :  '{processed}'")
    print()

Preprocessed Texts (first 6):

Original   :  'hello there'
Processed  :  'hello'

Original   :  'hi how are you'
Processed  :  'hi'

Original   :  'hey good morning'
Processed  :  'hey good morning'

Original   :  'good evening'
Processed  :  'good evening'

Original   :  'hi there what's up'
Processed  :  'hi 's'

Original   :  'hello bot'
Processed  :  'hello bot'



5. **Convert text to TF-IDF features**

**TF-IDF** (Term Frequency - Inverse Document Frequency) converts text into numerical vectors.
- **TF** -> How often a word appears in a sentence
- **IDF** -> How rare or important that word is across all sentences
- High TF-IDF score -> word is important and unique to that sentence

In [20]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(processed_texts)
y = labels

print("TF-IDF Feature Matrix Shape  :  ", X.shape)
print("Number of unique features     :  ", len(vectorizer.get_feature_names_out()))
print()
print("Feature names (vocabulary):")
print(list(vectorizer.get_feature_names_out()))

TF-IDF Feature Matrix Shape  :   (48, 70)
Number of unique features     :   70

Feature names (vocabulary):
['afternoon', 'air', 'arrive', 'bangalore', 'book', 'booking', 'bot', 'bye', 'cancel', 'care', 'catch', 'changed', 'chennai', 'date', 'day', 'delhi', 'delivery', 'evening', 'flight', 'fly', 'forecast', 'get', 'give', 'going', 'good', 'goodbye', 'hello', 'hey', 'hi', 'hot', 'kolkata', 'later', 'leaving', 'mind', 'morning', 'mumbai', 'need', 'next', 'nice', 'night', 'order', 'outside', 'package', 'placed', 'please', 'pune', 'purchase', 'rain', 'remove', 'reservation', 'reserve', 'seat', 'see', 'shipment', 'shipped', 'snow', 'soon', 'status', 'take', 'talk', 'tell', 'temperature', 'ticket', 'today', 'tomorrow', 'track', 'travel', 'update', 'want', 'weather']


6. **Split data into Train and Test sets**

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size  :  ", X_train.shape[0])
print("Test size   :  ", X_test.shape[0])

Train size  :   38
Test size   :   10


7. **Model 1: Naive Bayes Classifier for Intent Detection**

Naive Bayes is a simple probabilistic classifier based on Bayes' theorem.
It is fast, works well with small datasets, and is a popular baseline for text classification.

In [22]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)

print("Model 1: Naive Bayes")
print()
print("Accuracy  :  ", round(accuracy_score(y_test, nb_preds) * 100, 2), "%")
print()
print("Predictions vs Actual:")
print()
for actual, predicted in zip(y_test, nb_preds):
    print(f"Actual: {actual:<22}  Predicted: {predicted}")

Model 1: Naive Bayes

Accuracy  :   80.0 %

Predictions vs Actual:

Actual: get_weather             Predicted: cancel_order
Actual: check_order_status      Predicted: check_order_status
Actual: greeting                Predicted: greeting
Actual: book_flight             Predicted: book_flight
Actual: greeting                Predicted: greeting
Actual: goodbye                 Predicted: goodbye
Actual: goodbye                 Predicted: greeting
Actual: get_weather             Predicted: get_weather
Actual: check_order_status      Predicted: check_order_status
Actual: cancel_order            Predicted: cancel_order


8. **Model 2: Logistic Regression for Intent Detection**

Logistic Regression is a linear classification model.
It is widely used in NLP for text classification and generally outperforms Naive Bayes on larger datasets.

In [23]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

print("Model 2: Logistic Regression")
print()
print("Accuracy  :  ", round(accuracy_score(y_test, lr_preds) * 100, 2), "%")
print()
print("Predictions vs Actual:")
print()
for actual, predicted in zip(y_test, lr_preds):
    print(f"Actual: {actual:<22}  Predicted: {predicted}")

Model 2: Logistic Regression

Accuracy  :   60.0 %

Predictions vs Actual:

Actual: get_weather             Predicted: book_flight
Actual: check_order_status      Predicted: check_order_status
Actual: greeting                Predicted: greeting
Actual: book_flight             Predicted: book_flight
Actual: greeting                Predicted: get_weather
Actual: goodbye                 Predicted: goodbye
Actual: goodbye                 Predicted: greeting
Actual: get_weather             Predicted: get_weather
Actual: check_order_status      Predicted: check_order_status
Actual: cancel_order            Predicted: check_order_status


9. **Model 3: Support Vector Machine (SVM) for Intent Detection**

SVM finds the best decision boundary (hyperplane) that separates classes with the maximum margin.
LinearSVC is one of the best performing models for text classification tasks.

In [24]:
svm_model = LinearSVC(random_state=42, max_iter=1000)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)

print("Model 3: Support Vector Machine (LinearSVC)")
print()
print("Accuracy  :  ", round(accuracy_score(y_test, svm_preds) * 100, 2), "%")
print()
print("Predictions vs Actual:")
print()
for actual, predicted in zip(y_test, svm_preds):
    print(f"Actual: {actual:<22}  Predicted: {predicted}")

Model 3: Support Vector Machine (LinearSVC)

Accuracy  :   60.0 %

Predictions vs Actual:

Actual: get_weather             Predicted: book_flight
Actual: check_order_status      Predicted: check_order_status
Actual: greeting                Predicted: greeting
Actual: book_flight             Predicted: book_flight
Actual: greeting                Predicted: get_weather
Actual: goodbye                 Predicted: goodbye
Actual: goodbye                 Predicted: greeting
Actual: get_weather             Predicted: get_weather
Actual: check_order_status      Predicted: check_order_status
Actual: cancel_order            Predicted: check_order_status


10. **Compare all three models side by side**

In [25]:
models = {
    "Naive Bayes"         : round(accuracy_score(y_test, nb_preds)  * 100, 2),
    "Logistic Regression" : round(accuracy_score(y_test, lr_preds)  * 100, 2),
    "SVM (LinearSVC)"     : round(accuracy_score(y_test, svm_preds) * 100, 2)
}

print("Model Comparison for Intent Detection:")
print()
for model_name, acc in models.items():
    print(f"{model_name:<25}  :  Accuracy = {acc} %")

Model Comparison for Intent Detection:

Naive Bayes                :  Accuracy = 80.0 %
Logistic Regression        :  Accuracy = 60.0 %
SVM (LinearSVC)            :  Accuracy = 60.0 %


11. **Build the Intent Predictor Function using the best model (SVM)**

Method -> **predict_intent(user_input)** returns the detected intent

In [26]:
def predict_intent(user_input):
    processed = preprocess(user_input)
    vector = vectorizer.transform([processed])
    intent = svm_model.predict(vector)[0]
    return intent

test_inputs = [
    "hello how are you doing",
    "I want to fly to Mumbai tomorrow",
    "what will the weather be like today",
    "please cancel my booking",
    "where is my delivery",
    "bye see you"
]

print("Intent Predictions using SVM model:")
print()
for inp in test_inputs:
    print(f"Input   : '{inp}'")
    print(f"Intent  :  {predict_intent(inp)}")
    print()

Intent Predictions using SVM model:

Input   : 'hello how are you doing'
Intent  :  greeting

Input   : 'I want to fly to Mumbai tomorrow'
Intent  :  book_flight

Input   : 'what will the weather be like today'
Intent  :  get_weather

Input   : 'please cancel my booking'
Intent  :  cancel_order

Input   : 'where is my delivery'
Intent  :  check_order_status

Input   : 'bye see you'
Intent  :  goodbye



12. **Build a Simple Context-Aware Chatbot**

Context is maintained using a **conversation history list** that stores the last N intents and user messages.
The bot uses this context to give smarter, more relevant replies.

**Challenge demonstrated here:**
- Without context: the bot treats every message independently
- With context: the bot remembers what was said before and adjusts its response

In [27]:
responses = {
    "greeting"           : ["Hello! How can I help you today?",
                             "Hi there! What can I do for you?",
                             "Hey! How may I assist you?"],
    "goodbye"            : ["Goodbye! Have a great day!",
                             "See you later! Take care!",
                             "Bye! Feel free to come back anytime."],
    "book_flight"        : ["Sure! Where would you like to fly to?",
                             "I can help you book a flight. What is your destination?",
                             "Let me check available flights for you."],
    "get_weather"        : ["Let me fetch the current weather for you.",
                             "Sure! Which city's weather would you like to know?",
                             "I'll check the weather forecast right away."],
    "cancel_order"       : ["I can help you cancel your order. Please provide your order ID.",
                             "Sure, let me cancel that for you. What is your booking reference?",
                             "I'll process your cancellation immediately."],
    "check_order_status" : ["Please share your order ID and I will track it for you.",
                             "Let me check the status of your order.",
                             "I'll find the latest update on your delivery."]
}

# Context memory - stores last 3 (intent, message) pairs
conversation_history = []

def chatbot_reply(user_input):
    intent = predict_intent(user_input)
    conversation_history.append({"user": user_input, "intent": intent})

    # Context-aware logic: if user just booked a flight and now asks about weather
    if len(conversation_history) > 1:
        prev_intent = conversation_history[-2]["intent"]
        if prev_intent == "book_flight" and intent == "get_weather":
            return "Would you like the weather at your flight destination?"
        if prev_intent == "book_flight" and intent == "cancel_order":
            return "Do you want to cancel the flight you just booked?"

    reply = random.choice(responses[intent])
    return reply

print("Context-Aware Chatbot Simulation:")
print()

conversation = [
    "hello there",
    "book a flight to Delhi",
    "what is the weather today",
    "cancel my booking",
    "where is my order",
    "goodbye"
]

for user_msg in conversation:
    reply = chatbot_reply(user_msg)
    print(f"User   : {user_msg}")
    print(f"Bot    : {reply}")
    print()

Context-Aware Chatbot Simulation:

User   : hello there
Bot    : Hi there! What can I do for you?

User   : book a flight to Delhi
Bot    : Let me check available flights for you.

User   : what is the weather today
Bot    : Would you like the weather at your flight destination?

User   : cancel my booking
Bot    : Sure, let me cancel that for you. What is your booking reference?

User   : where is my order
Bot    : I'll find the latest update on your delivery.

User   : goodbye
Bot    : Bye! Feel free to come back anytime.



13. **Display Conversation History (Context Memory)**

In [28]:
print("Full Conversation History stored in context memory:")
print()
for i, turn in enumerate(conversation_history):
    print(f"Turn {i+1}  :  Message = '{turn['user']}'   |   Detected Intent = '{turn['intent']}'")

Full Conversation History stored in context memory:

Turn 1  :  Message = 'hello there'   |   Detected Intent = 'greeting'
Turn 2  :  Message = 'book a flight to Delhi'   |   Detected Intent = 'book_flight'
Turn 3  :  Message = 'what is the weather today'   |   Detected Intent = 'get_weather'
Turn 4  :  Message = 'cancel my booking'   |   Detected Intent = 'cancel_order'
Turn 5  :  Message = 'where is my order'   |   Detected Intent = 'check_order_status'
Turn 6  :  Message = 'goodbye'   |   Detected Intent = 'goodbye'


14. **Challenges in Building a Context-Aware Chatbot (Summary)**

Based on our implementation, the following key challenges are observed:

In [29]:
challenges = {
    "1. Ambiguous User Input"        : "Same words can mean different things in different contexts. E.g. 'cancel it' after booking a flight vs after ordering food.",
    "2. Context Window Management"   : "Knowing how many previous turns to remember. Too many = noise. Too few = lost context.",
    "3. Intent Overlap"              : "Intents like 'cancel_order' and 'check_order_status' share many keywords making classification harder.",
    "4. Out-of-Scope Inputs"         : "Users may say things the model was never trained on, leading to wrong or no intent prediction.",
    "5. Multi-Intent Messages"       : "A single message can have multiple intents. E.g. 'Book a flight and check the weather in Delhi'.",
    "6. Coreference Resolution"      : "Pronouns like 'it', 'there', 'that' refer to earlier mentions which are hard to resolve without context.",
    "7. Short and Noisy Text"        : "Users write with typos, slang, abbreviations which reduce model accuracy.",
    "8. Small Training Data"         : "With limited examples per intent, models struggle to generalise well to unseen inputs."
}

print("Challenges in Building a Context-Aware Chatbot:")
print()
for challenge, description in challenges.items():
    print(f"{challenge}")
    print(f"   -> {description}")
    print()

Challenges in Building a Context-Aware Chatbot:

1. Ambiguous User Input
   -> Same words can mean different things in different contexts. E.g. 'cancel it' after booking a flight vs after ordering food.

2. Context Window Management
   -> Knowing how many previous turns to remember. Too many = noise. Too few = lost context.

3. Intent Overlap
   -> Intents like 'cancel_order' and 'check_order_status' share many keywords making classification harder.

4. Out-of-Scope Inputs
   -> Users may say things the model was never trained on, leading to wrong or no intent prediction.

5. Multi-Intent Messages
   -> A single message can have multiple intents. E.g. 'Book a flight and check the weather in Delhi'.

6. Coreference Resolution
   -> Pronouns like 'it', 'there', 'that' refer to earlier mentions which are hard to resolve without context.

7. Short and Noisy Text
   -> Users write with typos, slang, abbreviations which reduce model accuracy.

8. Small Training Data
   -> With limited exampl

15. **Suitable NLP Models for Intent Detection (Summary)**

In [30]:
nlp_models = {
    "1. Naive Bayes + TF-IDF"           : "Simple, fast, good baseline. Best for small datasets and quick prototyping.",
    "2. Logistic Regression + TF-IDF"   : "Linear model, interpretable, handles multi-class well. Good for medium datasets.",
    "3. SVM (LinearSVC) + TF-IDF"       : "Best traditional ML model for text classification. High accuracy, handles high dimensions well.",
    "4. LSTM / BiLSTM"                  : "Deep learning model that captures word order and sequential patterns. Suitable for context-heavy tasks.",
    "5. BERT (Bidirectional Encoder)"   : "Transformer-based model pretrained on large corpora. Understands context from both directions. State-of-the-art for intent detection.",
    "6. DistilBERT"                     : "Lighter, faster version of BERT. Good for production chatbots with limited compute.",
    "7. Rasa NLU"                       : "Open-source framework designed specifically for chatbot intent detection and entity extraction.",
    "8. Dialogflow / AWS Lex"           : "Cloud-based NLU services with built-in intent detection. Best for enterprise chatbots without custom training."
}

print("Suitable NLP Models for Intent Detection:")
print()
for model_name, description in nlp_models.items():
    print(f"{model_name}")
    print(f"   -> {description}")
    print()

Suitable NLP Models for Intent Detection:

1. Naive Bayes + TF-IDF
   -> Simple, fast, good baseline. Best for small datasets and quick prototyping.

2. Logistic Regression + TF-IDF
   -> Linear model, interpretable, handles multi-class well. Good for medium datasets.

3. SVM (LinearSVC) + TF-IDF
   -> Best traditional ML model for text classification. High accuracy, handles high dimensions well.

4. LSTM / BiLSTM
   -> Deep learning model that captures word order and sequential patterns. Suitable for context-heavy tasks.

5. BERT (Bidirectional Encoder)
   -> Transformer-based model pretrained on large corpora. Understands context from both directions. State-of-the-art for intent detection.

6. DistilBERT
   -> Lighter, faster version of BERT. Good for production chatbots with limited compute.

7. Rasa NLU
   -> Open-source framework designed specifically for chatbot intent detection and entity extraction.

8. Dialogflow / AWS Lex
   -> Cloud-based NLU services with built-in intent de